
# Synthetic Project Risk Analysis

This project demonstrates an end-to-end data analysis workflow for a synthetic project risk dataset. We will explore the data, visualize key metrics, and build predictive models to classify project risk levels. The goal is to help business analysts, program managers, and data analysts understand how data science techniques can be applied to project management and risk assessment.

The dataset contains 1,000 synthetic project records with features such as budget, project duration, team size, scope changes, quality incidents, stakeholder count, team experience, vendor reliability, and complexity score. Each project is labeled with a **Risk Level** category (`Low`, `Medium`, or `High`).


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline


In [ ]:

# Load dataset
file_path = 'synthetic_project_risk_data.csv'
df = pd.read_csv(file_path)

# Display first few rows
df.head()


In [ ]:

# Summary statistics and basic information
df.describe(include='all')



## Exploratory Data Analysis

Let's explore the distribution of numerical features and the balance of the target classes (`Risk_Level`).


In [ ]:

# Histogram for numeric features
numeric_cols = df.drop(columns=['Risk_Level']).columns

plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_cols):
    plt.subplot(3, 3, i + 1)
    sns.histplot(df[col], kde=True, bins=20, color='skyblue')
    plt.title(col)
plt.tight_layout()
plt.show()


In [ ]:

# Bar plot for risk level distribution
plt.figure(figsize=(5,4))
sns.countplot(data=df, x='Risk_Level', palette='viridis')
plt.title('Risk Level Distribution')
plt.xlabel('Risk Level')
plt.ylabel('Count')
plt.show()


In [ ]:

# Compute correlation matrix for numeric variables
corr = df.drop(columns=['Risk_Level']).corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()



## Predictive Modeling

We will build models to predict the `Risk_Level` of a project. We'll encode the target variable and use two algorithms: Logistic Regression and Random Forest. We'll evaluate the models using classification metrics.


In [ ]:

# Encode target variable
X = df.drop(columns=['Risk_Level'])
y = df['Risk_Level']

# One-hot encode the categorical target variable for evaluation
y_encoded = y.map({'Low':0, 'Medium':1, 'High':2})

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y)

# Numeric features (all features here are numeric)
numeric_features = X.columns

# Preprocess: scale numeric features
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)
    ])


In [ ]:

# Logistic Regression pipeline
log_reg = Pipeline(steps=[('preprocess', preprocess),
                     ('classifier', LogisticRegression(max_iter=1000))])

# Train the model
log_reg.fit(X_train, y_train)

# Predict on test set
y_pred_lr = log_reg.predict(X_test)

# Evaluation
print("Logistic Regression Classification Report:
")
print(classification_report(y_test, y_pred_lr, target_names=['Low','Medium','High']))

print("Confusion Matrix:
")
print(confusion_matrix(y_test, y_pred_lr))


In [ ]:

# Random Forest pipeline
rf = Pipeline(steps=[('preprocess', preprocess),
                ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))])

# Train the model
rf.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf.predict(X_test)

# Evaluation
print("Random Forest Classification Report:
")
print(classification_report(y_test, y_pred_rf, target_names=['Low','Medium','High']))

print("Confusion Matrix:
")
print(confusion_matrix(y_test, y_pred_rf))


In [ ]:

# Extract feature importances
rf_model = rf.named_steps['classifier']
importances = rf_model.feature_importances_

importance_df = pd.DataFrame({'Feature': numeric_features, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='magma')
plt.title('Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.show()



## Conclusion

In this analysis, we generated a synthetic dataset representing different project characteristics and their associated risk levels. We performed exploratory data analysis to understand the distributions and correlations among features, and we built predictive models to classify projects into **Low**, **Medium**, or **High** risk categories.

Both Logistic Regression and Random Forest models achieved reasonable performance on this synthetic dataset. The Random Forest model provided higher accuracy and revealed the relative importance of features, highlighting factors like **Scope Changes**, **Quality Incidents**, and **Complexity Score** as key contributors to project risk.

You can extend this analysis by experimenting with other algorithms, tuning hyperparameters, or applying the workflow to real-world project data. This project serves as a template for applying data science techniques to project management and risk assessment.
